In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()

src_path = repo_root / "../"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from doc_parsers.pdfParser import pdfParser
from doc_parsers.mdParser import md_parser


In [2]:
BUNDLE_DIR = Path("../examples")  # where the bundle PDFs are
PDFS = [
    BUNDLE_DIR / "bundle_SOP_AFW_P101A.pdf",
    BUNDLE_DIR / "bundle_CR_2026_00123.pdf",
    BUNDLE_DIR / "bundle_WO_2026_04567.pdf",
    BUNDLE_DIR / "bundle_ECA_2026_0007.pdf",
]

DEST_ROOT = Path("parsed_docs")
DEST_ROOT.mkdir(parents=True, exist_ok=True)

PDFS, DEST_ROOT

([PosixPath('../examples/bundle_SOP_AFW_P101A.pdf'),
  PosixPath('../examples/bundle_CR_2026_00123.pdf'),
  PosixPath('../examples/bundle_WO_2026_04567.pdf'),
  PosixPath('../examples/bundle_ECA_2026_0007.pdf')],
 PosixPath('parsed_docs'))

In [3]:
# Choose one PDF
pdf_path = PDFS[0]

# Since pdfParser wants (home_folder, red_filepath), set home_folder and make pdf relative
home_folder = str(BUNDLE_DIR)
red_filepath = pdf_path.relative_to(BUNDLE_DIR).as_posix()

doc_index = pdfParser(
    home_folder=home_folder,
    red_filepath=red_filepath,
    destination_folder=str(DEST_ROOT),
    text2markdown="marker",
    tableParser="marker",          # or "pdfplumber" :contentReference[oaicite:16]{index=16}
    classification="internal",
    ingest_id="demo_ingest_001",
    source_path=red_filepath,      # stored in document_index :contentReference[oaicite:17]{index=17}
)

structured = md_parser(
    document_index=doc_index,
    destination_folder=None,       # defaults to base folder above text_md_path :contentReference[oaicite:18]{index=18}
    mbse_entities=None,
    nureg_section_ids=None
)

doc_index.keys(), structured.keys()

2026-03-24 16:52:17 | INFO | pdfParser | Rendering PDF to Markdown via Marker: ../examples/bundle_SOP_AFW_P101A.pdf
2026-03-24 16:52:19,758 [WARNING] surya: `TableRecEncoderDecoderModel` is not compatible with mps backend. Defaulting to cpu instead
Running OCR Error Detection: 100%|██████████| 1/1 [00:00<00:00,  1.39it/s]
Detecting bboxes: 0it [00:00, ?it/s]
Recognizing tables: 100%|██████████| 1/1 [00:00<00:00,  1.29it/s]
Detecting bboxes: 0it [00:00, ?it/s]
2026-03-24 16:52:28 | INFO | pdfParser | Extracting tables via Marker: ../examples/bundle_SOP_AFW_P101A.pdf
2026-03-24 16:52:28,988 [WARNING] surya: `TableRecEncoderDecoderModel` is not compatible with mps backend. Defaulting to cpu instead
Running OCR Error Detection: 100%|██████████| 1/1 [00:00<00:00, 125.43it/s]
Detecting bboxes: 0it [00:00, ?it/s]
Recognizing tables: 100%|██████████| 1/1 [00:00<00:00,  1.34it/s]
Detecting bboxes: 0it [00:00, ?it/s]
2026-03-24 16:52:34 | INFO | pdfParser | Parsing complete for: ../examples/bund

(dict_keys(['doc_id', 'doc_type', 'authority_level', 'source_path', 'doc_name', 'ingest_id', 'text_md_path', 'figures', 'tables_paths', 'tables_index', 'metadata_path', 'source', 'content_hash', 'doc_version', 'extraction', 'timestamp', 'classification']),
 dict_keys(['doc_id', 'sections', 'tables', 'figures', 'timestamp', 'provenance', 'source_path', 'doc_name', 'ingest_id', 'chunks_jsonl_path', 'structured_output_path']))

In [4]:
import json

doc_dirs = [p for p in DEST_ROOT.iterdir() if p.is_dir()]
doc_dirs

[PosixPath('parsed_docs/a854819ee608')]

In [5]:
# pick one doc folder
d = doc_dirs[0]
index_dir = d / "index"

# 3.2 Open a structured output and view section titles

structured_path = next(index_dir.glob("*_structured_output.json"))
chunks_path = next(index_dir.glob("*_chunks.jsonl"))

structured = json.loads(structured_path.read_text(encoding="utf-8"))

structured_path, chunks_path, structured.keys()

(PosixPath('parsed_docs/a854819ee608/index/a854819ee608_structured_output.json'),
 PosixPath('parsed_docs/a854819ee608/index/a854819ee608_chunks.jsonl'),
 dict_keys(['doc_id', 'sections', 'tables', 'figures', 'timestamp', 'provenance', 'source_path', 'doc_name', 'ingest_id', 'chunks_jsonl_path', 'structured_output_path']))

In [6]:
# 3.3 Load chunks.jsonl and filter what gets embedded in Chroma
chunks = [json.loads(line) for line in chunks_path.read_text(encoding="utf-8").splitlines() if line.strip()]

# what chunk types do we have?
from collections import Counter
Counter([c["type"] for c in chunks])

# get only those intended for vector store
vector_chunks = [c for c in chunks if c.get("index_in_vector_store")]

len(vector_chunks), vector_chunks[0].keys()

(9,
 dict_keys(['type', 'granularity', 'chunk_id', 'doc_id', 'doc_type', 'doc_key', 'section_title', 'section_role', 'heading_level', 'text', 'raw_text', 'keywords', 'mentions_component_ids', 'references_standard_refs', 'nureg_section_ids', 'page_start', 'page_end', 'provenance', 'classification', 'index_in_vector_store', 'index_in_graph', 'source_path', 'doc_name', 'ingest_id', 'content_hash']))

In [7]:
# Summarization
import os
os.environ["OLLAMA_BASE_URL"] = "http://localhost:11434"
os.environ["OLLAMA_MODEL"] = "mistral:latest"     # or your local model
os.environ["OLLAMA_NUM_CTX"] = "8192"

In [8]:
from pathlib import Path

DEST_ROOT = Path("parsed_docs/fc4d8015d284")
chunks_files = sorted(DEST_ROOT.glob("index/*_chunks.jsonl"))
chunks_files

[]

In [10]:
#Run augmentation on one file
from ner.augment_chunks import augment_chunks_with_structured_summaries

stats = augment_chunks_with_structured_summaries(
    chunks_files[0],
    model=None,                  # uses env OLLAMA_MODEL
    overwrite=False,
    summarize_granularities=("section", "paragraph"),
    only_indexable=True,
)

stats

ModuleNotFoundError: No module named 'causal_condition_adapter'